# Topic: Secure API Engineering - CORS policies, secure HTTP headers (HSTS/CSP), and token bucket rate limiters
 
## 1. SECURE API HEADERS & POLICY CONTROLS
- **CORS (Cross-Origin Resource Sharing)**:
  - An HTTP-header-based mechanism that allows a server to indicate any origins (domain, scheme, or port) other than its own from which a browser should permit loading resources.
  - **Security Rule**: Avoid `Access-Control-Allow-Origin: *` in production APIs that require cookies or private session auth tokens.
- **Secure HTTP Headers**:
  - **HSTS (HTTP Strict Transport Security)**: Forces browsers to communicate with the API server exclusively using secure HTTPS connections.
  - **CSP (Content Security Policy)**: RESTRICTS the sources of scripts, styles, and assets that the browser is allowed to execute, neutralizing XSS.
  - **X-Frame-Options**: Prevents **Clickjacking** by restricting whether a browser is allowed to render the site inside `<iframe>` tags.
 
## 2. RATE LIMITING: THE TOKEN BUCKET ALGORITHM
- **Purpose**: Prevents Denial of Service (DoS) and automated API scraping.
- **Token Bucket Logic**:
  - A bucket has a maximum capacity $C$ and is filled with tokens at a constant rate $r$ tokens/second.
  - Each incoming API request consumes 1 token.
  - If the bucket contains tokens, the request is allowed. If the bucket is empty, the request is blocked (`HTTP 429 Too Many Requests`).
 
---


In [ ]:
import time

class TokenBucketRateLimiter:
    """Implements the Token Bucket rate limiting algorithm."""
    def __init__(self, capacity, fill_rate):
        self.capacity = float(capacity)
        self.fill_rate = float(fill_rate)  # Tokens per second
        self.tokens = float(capacity)
        self.last_update = time.time()

    def _refill_tokens(self):
        """Refills tokens based on elapsed time."""
        now = time.time()
        elapsed = now - self.last_update
        
        # Calculate new token count
        refilled_tokens = elapsed * self.fill_rate
        self.tokens = min(self.capacity, self.tokens + refilled_tokens)
        self.last_update = now

    def allow_request(self):
        """Checks if a request can consume a token. Returns True (Allow) or False (Block)."""
        self._refill_tokens()
        
        if self.tokens >= 1.0:
            self.tokens -= 1.0
            return True
        return False



In [ ]:
# Testing the Rate Limiter
print("--- Testing Token Bucket Rate Limiter ---")
# Capacity: 3 requests. Fill rate: 1 token per second.
limiter = TokenBucketRateLimiter(capacity=3, fill_rate=1.0)

# Simulate rapid bursts
for i in range(1, 6):
    allowed = limiter.allow_request()
    print(f"Request {i} | Allowed? {allowed} | Remaining Tokens: {limiter.tokens:.2f}")
# Expected: Requests 1, 2, 3 are allowed (burst capacity consumed).
# Requests 4, 5 are blocked!



In [ ]:
# Wait for refill
print("\n--- Waiting for 1.5 seconds to refill tokens ---")
time.sleep(1.5)

# Try again
for i in range(6, 8):
    allowed = limiter.allow_request()
    print(f"Request {i} | Allowed? {allowed} | Remaining Tokens: {limiter.tokens:.2f}")
# Expected: Request 6 is allowed (due to refill), Request 7 is blocked.
